# Basic Auto-Instrumentation Setup for Ollama

> **Before we begin, make sure that you are running an observability suite (Grafana etc.), and that _local Ollama is ready_.**  
> See the project root's `README.md` for more tips on that.

|||
|--|--|
|**Main Goal**| Experience the basic of **OpenTelemetry tracing for Ollama API**. |
|**Approach**| Here we setup an **auto-instrumented tracing configuration for Ollama usage**. Some manual configuration of the OpenTelemetry transports is still needed, but LLM API usage is wrapped in tracing immediately.|

In [1]:
!cd .. && pip install --group "runtime" --group "ollama"

## Auto-Instrumented Ollama Usage

> **Load local environment variables**

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

> **We configure logging to see OpenTelemetry external transport**

In [3]:
import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.DEBUG)
logger = logging.getLogger("opentelemetry")
logger.setLevel(logging.DEBUG)

> **Here we set the tracer up with two providers: `console` and `otlp` (external collector)**

In [4]:
import os

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
otlp_endpoint = os.environ["OTEL_EXPORTER_OTLP_TRACES_ENDPOINT"]
provider.add_span_processor(SimpleSpanProcessor(OTLPSpanExporter(endpoint=otlp_endpoint)))

trace.set_tracer_provider(provider)

> **This code automatically instruments all subsequent calls to the Ollama APIs**

In [5]:
from opentelemetry.instrumentation.ollama import OllamaInstrumentor

OllamaInstrumentor().instrument()

> **This is some basic "business" logic we want to trace**

In [6]:
from ollama import Client
from pydantic import BaseModel

DEFAULT_CLIENT = Client(host='http://localhost:11434')

class Flower(BaseModel):
    vernacular_name: str
    genus: str
    tribe: str
    subfamily: str
    family: str
    order: str
    kingdom: str
    custom_elaborate_summary: str
    brief_use_example_sentences: list[str]

def _describe_a_flower(flower: str, llm_client: Client) -> Flower:
    request = f"Tell me about a flower '{flower}', and its scientific classification."
    response = llm_client.chat(
        model='llama3.2:1b',
        messages=[{'role': 'user', 'content': request}],
        format=Flower.model_json_schema(),
        stream=False,
    )
    return Flower.model_validate_json(response.message.content)

def describe_a_flower(flower:str) -> Flower:
    return _describe_a_flower(flower, DEFAULT_CLIENT)

> **Running the code therefore records the span and pushes it to the configured trace providers**

In [7]:
describe_a_flower("Rose")

DEBUG:httpcore.connection:connect_tcp.started host='localhost' port=11434 local_address=None timeout=None socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x112464590>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Content-Type', b'application/json; charset=utf-8'), (b'Date', b'Wed, 28 Oct 2026 12:01:42 GMT'), (b'Transfer-Encoding', b'chunked')])
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
DEBUG:httpcore.http11:receive_response_body.started request=<Request [b'POST']>
DEBUG:httpcore.http11:r

Flower(vernacular_name='Rose', genus='Rosa', tribe='Rosae', subfamily='Rosoideae', family='Rosaceae', order='Rosales', kingdom='Plantae', custom_elaborate_summary='Rose is a genus of flowering plants in the rose family, commonly known as the rose. The genus rose consists of over 150 species, with the most well-known being the sweet rose, the damask rose, and the climbing rose. The scientific classification of the rose is as follows: Genus: Rosa Species: R. gallica, R. centifolia, R. damascena, and many others. Kingdom: Plantae Order: Rosales Family: Rosaceae Genus: Rosa Classification is based on the botanical definition, which distinguishes the rose from other flowers by its: Epithelium: The outer layer of the flower, which is made of tightly packed cells. Petal: The modified leaf that is usually colorful and fragrant. Stamens: The male reproductive organs of the flower, which produce pollen. Pistil: The female reproductive organ of the flower, which contains the ovary and seeds. Clas

> **As a result, with an observability suite (Grafana and etc) running, you should find the traces in the platform!**  
> **For the LLM-specific tracing details, see attributes with `gen_ai` and `llm` prefixes.**  
> There may be a small delay before you receive the traces.